# rev-03 scoring on Kaggle: the two trained correctors on the validation and BL test pages

Runs `correction/score_trained_corrector.py` for ByT5-small and BanglaByT5, one model per GPU, first on the validation pages
(a check against the training log, split `val`) and then on the BL test pages (split `test_bl`). See `context/decisions.md` D76.
Nothing is trained or tuned here. Run it once; the test numbers are the ones for the paper.

**Before running:**
1. Session options: **Accelerator = GPU T4 x2**, Internet = on.
2. Add data (all already in your account):
   - the output of notebook `bengali-rev-03-train-both` (holds `train/<model>/best`),
   - the outputs of `bengali-rev-01a-ocr-test-pages`, `bengali-rev-01b-ocr-trainval-1` and `bengali-rev-01c-ocr-trainval-2` (the `*_region.jsonl` files),
   - the dataset `bengali-revision-pages` (holds `val.txt` and `test_bl.txt`).
3. No Hugging Face token is needed (both models are public).
4. Save Version, then "Save & Run All (Commit)".

When it finishes, download `rev03_score.zip` from the version's Output tab. It holds `summary.json`, `summary.txt` and `pages.jsonl` for each model and split.
`pages.jsonl` contains page text; keep it out of Git (`results/` is ignored).


In [ ]:
import glob, json, os, shutil, subprocess, sys, time

REPO_DIR = "/tmp/bengali-postocr-sllm"
OUT_DIR = "/kaggle/working/rev03_score"
os.makedirs(OUT_DIR, exist_ok=True)
SETUP_LOG = f"{OUT_DIR}/setup.log"

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    with open(SETUP_LOG, "a") as f:
        f.write(f"\n$ {cmd}\nexit {r.returncode}\n{r.stdout[-3000:]}{r.stderr[-3000:]}")
    print(("ok   " if r.returncode == 0 else "FAIL ") + cmd)
    return r.returncode == 0

if not os.path.isdir(REPO_DIR):
    sh(f"git clone -q https://github.com/Ariffurrhmn/bengali-postocr-sllm.git {REPO_DIR}")
assert os.path.isdir(REPO_DIR), "git clone failed: is Internet on in Session options? See rev03_score/setup.log"
# the scoring script was added in commit 0af985f
assert os.path.exists(f"{REPO_DIR}/correction/score_trained_corrector.py"), "clone is older than the scoring script"
if not sh("pip install -q transformers==5.19.0 accelerate sentencepiece jiwer rapidfuzz scipy"):
    sh("pip install -q -U transformers accelerate sentencepiece jiwer rapidfuzz scipy")

commit = subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout
import torch, transformers
env = (f"repo commit: {commit}python: {sys.version.split()[0]}\ntorch: {torch.__version__}\n"
       f"cuda: {torch.version.cuda}\ntransformers: {transformers.__version__}\n"
       f"gpus: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}\n")
open(f"{OUT_DIR}/environment.txt", "w").write(env)
print(env)
assert torch.cuda.device_count() == 2, "Set Accelerator to GPU T4 x2"


In [ ]:
def one(pattern, what):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, f"not found: {what} ({pattern}); attach the input listed at the top"
    return hits[0]

WEIGHTS = {m: os.path.dirname(one(f"/kaggle/input/**/train/{m}/best/model.safetensors", f"{m} weights")) for m in ("byt5-small", "banglabyt5")}
REGION = [one(f"/kaggle/input/**/{n}", n) for n in ("shard_a_region.jsonl", "shard_b_region.jsonl", "shard_c_region.jsonl")]
SPLIT_DIR = os.path.dirname(one("/kaggle/input/**/test_bl.txt", "test_bl.txt"))
assert os.path.exists(f"{SPLIT_DIR}/val.txt"), "val.txt is not next to test_bl.txt"
print(json.dumps({"weights": WEIGHTS, "region": REGION, "split_dir": SPLIT_DIR}, indent=1))
for p in REGION:
    print(os.path.basename(p), sum(1 for _ in open(p, encoding="utf-8")), "pages")


In [ ]:
# One model per GPU, in parallel. Each runs val first (check against the training log), then test_bl.
procs = []
for gpu, model in enumerate(("byt5-small", "banglabyt5")):
    cmds = []
    for split in ("val", "test_bl"):
        out = f"{OUT_DIR}/{model}_{split}"
        cmds.append(f"{sys.executable} score_trained_corrector.py --model {model} --weights {WEIGHTS[model]} "
                    f"--ocr-region {' '.join(REGION)} --split-dir {SPLIT_DIR} --split {split} --out-dir {out} "
                    f"> {OUT_DIR}/{model}_{split}.log 2>&1")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONIOENCODING="utf-8")
    procs.append((model, subprocess.Popen(" ; ".join(cmds), shell=True,
                                          cwd=f"{REPO_DIR}/correction", env=env)))
t0 = time.time()
for model, p in procs:
    p.wait()
    print(f"{model} finished with exit {p.returncode} after {(time.time() - t0) / 60:.1f} min")


In [ ]:
for model in ("byt5-small", "banglabyt5"):
    for split in ("val", "test_bl"):
        f = f"{OUT_DIR}/{model}_{split}/summary.txt"
        print("=" * 20, model, split)
        print(open(f, encoding="utf-8").read() if os.path.exists(f) else "MISSING: see " + f"{OUT_DIR}/{model}_{split}.log")
shutil.make_archive("/kaggle/working/rev03_score", "zip", "/kaggle/working", "rev03_score")
print("wrote /kaggle/working/rev03_score.zip")
